# 05 · Deployment & Consumption

Following Géron Ch. 2 — *"Launch, Monitor and Maintain Your System"*.

> *"Your model is ready to be deployed. You need to plug it into a production system."*

This project exposes predictions through **three interfaces**:

| Interface | Use case | How |
|---|---|---|
| **REST API** (FastAPI) | Programmatic consumption / CI | `make api` |
| **Streamlit dashboard** | Internal data team | `make streamlit` |
| **Telegram Bot** | CFO / executive query on mobile | `make bot` |

![Telegram demo](../assets/telegram_rossmann.gif)

## 1. REST API (FastAPI)

### Start the server

```bash
make api
# or: PYTHONPATH=src uvicorn rossmann_store_sales.api:app --reload
```

### Endpoints

| Method | Path | Description |
|---|---|---|
| `GET` | `/health` | Health check |
| `POST` | `/rossmann/predict` | Predict sales for records |

In [ ]:
# Live test against the running API
import json, requests, pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

API_URL = 'http://127.0.0.1:8000'

# 1. Health check
try:
    r = requests.get(f'{API_URL}/health', timeout=3)
    print('Health:', r.json())
except Exception as e:
    print(f'API not running — start with `make api`. ({e})')

In [ ]:
# 2. Batch prediction from sample file
sample = pd.read_csv(ROOT / 'data/sample/store_1_scoring.csv')

try:
    resp = requests.post(
        f'{API_URL}/rossmann/predict',
        json={'records': sample.to_dict(orient='records')},
        timeout=30,
    )
    resp.raise_for_status()
    result = pd.DataFrame(resp.json())
    print(f'Predicted {len(result)} rows. Total forecast: {result["prediction"].sum():,.2f}')
    result[['store', 'date', 'prediction']].head(10)
except Exception as e:
    print(f'Skipped — start the API first. ({e})')

## 2. Streamlit dashboard

```bash
make streamlit
# or: PYTHONPATH=src streamlit run app/streamlit_app.py
```

- Upload any CSV with store records, or use the bundled sample.
- Click **Forecast** to call the API and display the time-series chart.

## 3. Telegram Bot

The bot makes the model accessible to non-technical stakeholders on mobile.

### Setup

```bash
# 1. Create a bot with @BotFather on Telegram → copy the token
# 2. Copy .env.example to .env and fill in TELEGRAM_TOKEN
cp .env.example .env

# 3. Start the API (terminal 1)
make api

# 4. Start the bot (terminal 2)
make bot
```

### Usage

| User sends | Bot responds |
|---|---|
| `/start` or `/help` | Instructions |
| `/1` | 6-week forecast for store 1 + chart |
| `/42` | 6-week forecast for store 42 + chart |

## 4. Local simulation — no real Telegram needed

In [ ]:
# Simulate a Telegram webhook payload directly
# (the bot must be running and the API must be up)
import requests, json

BOT_URL = 'http://127.0.0.1:5000/rossmann/bot'
STORE_ID = 1

payload = {
    'message': {
        'chat': {'id': 999999},
        'text': f'/{STORE_ID}',
    }
}

try:
    r = requests.post(BOT_URL, json=payload, timeout=30)
    print('Bot response status:', r.status_code)
    print('(Check terminal 2 for the forecast output)')
except Exception as e:
    print(f'Bot not running — start with `make bot`. ({e})')

## 5. Webhook deployment (public URL)

For the Telegram bot to work with real users, the Flask app must be reachable  
from the internet via HTTPS.  Free options: **Render**, **Railway**, **Fly.io**, **Heroku**.

```bash
# After deploying, register the URL with Telegram:
make webhook WEBHOOK_URL=https://your-app.onrender.com

# or manually:
python scripts/set_telegram_webhook.py --url https://your-app.onrender.com/rossmann/bot
```

The `Procfile.api` and `Procfile.bot` files at the repo root are ready for  
Heroku / Render deployments — one process per service.

## 6. Environment variables

Copy `.env.example` → `.env` and fill in:

| Variable | Description |
|---|---|
| `TELEGRAM_TOKEN` | Token from @BotFather |
| `ROSSMANN_API_URL` | Prediction endpoint (default: `http://127.0.0.1:8000/rossmann/predict`) |
| `ROSSMANN_TEST_PATH` | Path to `test.csv` (default: `data/raw/test.csv`) |
| `ROSSMANN_STORE_PATH` | Path to `store.csv` (default: `data/raw/store.csv`) |